# AI-Publisher Docker Build (Disk- Safe)\n\nHer model icin sirayla:\n1. HF agirliklarini indir (Drive cache)\n2. Docker build (Kaniko)\n3. Drive'a .tar.gz olarak kaydet\n4. Disk alanini temizle (HF cache + Kaniko artifact)\n\nBoylece disk dolmaz — her modelin build+save islemi bitince temizlik yapilir.\n\n**NOT:** Build islemi ~5-15 dakika/smodel sürer. Toplam ~2-4 saat.

## Hucre 1: Drive Baglantisi + Setup

In [ ]:
import os\nimport subprocess\nimport sys\nimport shutil\nimport time\nfrom pathlib import Path\n\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\n# Konfigurasyon\nDRIVE_DIR = Path(\"/content/drive/MyDrive/Colab Notebooks/docker/images\")\nHF_CACHE_DIR = Path(\"/content/drive/MyDrive/Colab Notebooks/docker/hf_cache\")\nWORKSPACE = Path(\"/content/AI-Publisher/colab_docker\")\nos.environ['HF_HOME'] = str(HF_CACHE_DIR)\n\ntry:\n    from google.colab import userdata\n    hf_token = userdata.get('HF_TOKEN')\n    if hf_token:\n        os.environ['HF_TOKEN'] = hf_token\n        print('HF_TOKEN loaded successfully!')\nexcept Exception as e:\n    print(f'HF_TOKEN could not be loaded from Colab secrets: {e}')\n\nDRIVE_DIR.mkdir(parents=True, exist_ok=True)\nHF_CACHE_DIR.mkdir(parents=True, exist_ok=True)\nos.makedirs(WORKSPACE, exist_ok=True)\n\nprint(f\"DRIVE_DIR : {DRIVE_DIR}\")\nprint(f\"HF_CACHE : {HF_CACHE_DIR}\")\nprint(f\"WORKSPACE: {WORKSPACE}\")\n\n# Kaniko kurulum kontrol\nKANIKO = None\nfor p in [\"/usr/local/bin/kaniko\", \"/kaniko/executor\"]:\n    if Path(p).exists():\n        KANIKO = p\n        break\n\nif not KANIKO:\n    print(\"Kaniko indiriliyor...\")\n    subprocess.run(\n        \"wget -q https://github.com/GoogleContainerTools/kaniko/releases/download/v1.19.2/kaniko-executor.tar.gz \"\n        \"-O /tmp/kaniko.tar.gz && tar -xzf /tmp/kaniko.tar.gz -C /tmp && mv /tmp/kaniko /usr/local/bin/kaniko && chmod +x /usr/local/bin/kaniko\",\n        shell=True, check=True\n    )\n    KANIKO = \"/usr/local/bin/kaniko\"\n\nprint(f\"Kaniko : {KANIKO}\")\n\n# Local registry baslat\ndef cmd(c, check=True, capture=False, timeout=None):\n    r = subprocess.run(c, shell=True, text=True, capture_output=capture, timeout=timeout)\n    if capture:\n        r._out = r.stdout.strip() if r.stdout else \"\"\n    if check and r.returncode != 0:\n        print(f\"[HATA] {c}\")\n        if r.stderr: print(r.stderr[:500])\n        sys.exit(1)\n    return r\n\ntry:\n    r = cmd(\"curl -s -f http://localhost:5000/v2/\", check=False, capture=True)\n    print(\"Registry : localhost:5000 calisiyor\")\nexcept:\n    print(\"Registry baslatiliyor...\")\n    cmd(\"pkill -f 'registry serve' 2>/dev/null; nohup registry serve /etc/docker/registry/config.yml > /tmp/registry.log 2>&1 &\")\n    time.sleep(3)\n    print(\"Registry : localhost:5000 hazir\")

## Hucre 2: HF Agirlik On-Indirme (Model Bazli)

In [ ]:
def disk_free(path=\"/workspace\"):\n    \"\"\"Workspace'de kalan GB.\"\"\"\n    r = subprocess.run(f\"df -BG {path} | tail -1\", shell=True, text=True, capture_output=True)\n    parts = r.stdout.strip().split()\n    # 4th column = available\n    available_gb = int(parts[3].replace(\"G\", \"\"))\n    return available_gb\n\ndef download_hf(repo_id, subpath=None):\n    \"\"\"Tek HF model indir. Cache varsa skip.\"\"\"\n    cache_key = repo_id.replace(\"/\", \"--\")\n    cache_marker = HF_CACHE_DIR / f\"models--{cache_key}\"\n    if cache_marker.exists():\n        print(f\"  [CACHE] {repo_id} — zaten var, atleniyor\")\n        return True\n    \n    print(f\"  [DOWN] {repo_id}...\")\n    free = disk_free()\n    print(f\"  [DISK] {free}GB free\")\n    \n    if free < 6:\n        print(f\"  [WARN] Disk kritik ({free}GB). Temizlik yapiliyor...\")\n        subprocess.run(\"rm -rf /workspace/hf_cache/* /tmp/kaniko-* 2>/dev/null\", shell=True)\n        time.sleep(2)\n        free = disk_free()\n        if free < 6:\n            print(f\"  [HATA] Disk yetersiz ({free}GB). Durduruluyor.\")\n            return False\n    \n    try:\n        if subpath:\n            cmd = f\"\"\"python -c \"\nfrom huggingface_hub import snapshot_download, os\npath = snapshot_download(\n    repo_id='{repo_id}',\n    cache_dir=os.environ.get('HF_HOME', '{HF_CACHE_DIR}'),\n    allow_patterns=['{subpath}/*'],\n    token=os.environ.get('HF_TOKEN') or None,\n    ignore_patterns=['*.msgpack','*.h5','*.tflite','*.onnx','*.pb'],\n)\nprint(f'[OK] {path}')\n\"\"\"\n        else:\n            cmd = f\"\"\"python -c \"\nfrom huggingface_hub import snapshot_download, os\npath = snapshot_download(\n    repo_id='{repo_id}',\n    cache_dir=os.environ.get('HF_HOME', '{HF_CACHE_DIR}'),\n    token=os.environ.get('HF_TOKEN') or None,\n    ignore_patterns=['*.msgpack','*.h5','*.tflite','*.onnx','*.pb'],\n)\nprint(f'[OK] {path}')\n\"\"\"\n        r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=3600)\n        if r.returncode == 0:\n            print(f\"  [OK] {repo_id} — {disk_free()}GB kaldi\")\n            return True\n        else:\n            print(f\"  [HATA] {repo_id}\")\n            if r.stderr: print(r.stderr[:300])\n            return False\n    except subprocess.TimeoutExpired:\n        print(f\"  [HATA] Timeout — {repo_id}\")\n        return False\n\n# HF agirliklari — her model on-cesitinde indirilecek modeller\n# Buyuk modeller once (fail fast icin)\nALL_REPOS = [\n    # (repo_id, subpath veya None)\n    (\"Lightricks/LTX-Video\", None),\n    (\"genmo/mochi-1-preview\", None),\n    (\"hunyuanvideo-community/HunyuanVideo\", None),\n    (\"Wan-AI/Wan2.1-I2V-14B-480P\", None),\n    (\"Wan-AI/Wan2.1-T2V-1.3B\", None),\n    (\"THUDM/CogVideoX-5b\", None),\n    (\"THUDM/CogVideoX-5b-I2V\", None),\n    (\"THUDM/CogVideoX-2b\", None),\n    (\"stabilityai/stable-video-diffusion-img2vid-xt\", None),\n    (\"guoyww/animatediff-motion-adapter-v1-5-2\", None),\n    (\"frankjoshua/toonyou_beta6\", None),\n    (\"cerspense/zeroscope_v2_576w\", None),\n    (\"DynamiCrafter/dynamicrafter_512_interp_512\", None),\n    (\"hexgrad/Kokoro-82M\", None),\n    (\"charactr/vocos-mel-24khz\", None),\n    (\"cvssp/audioldm2\", None),\n    (\"coqui/XTTS-v2\", \"tts_models/multilingual/multi-dataset/xtts_v2\"),\n    (\"Systran/faster-whisper-small\", None),\n    (\"Systran/faster-whisper-medium\", None),\n    (\"Systran/faster-whisper-large-v3\", None),\n    (\"openai/whisper-small\", None),\n    (\"openai/whisper-large-v3\", None),\n    (\"stabilityai/stable-diffusion-xl-base-1.0\", None),\n]\n\nprint(f\"Toplam {len(ALL_REPOS)} HF repo. On-indirme basliyor...\")\nprint(f\"Drive cache: {HF_CACHE_DIR}\")\nprint()\n\nok_count = 0\nfail_count = 0\nfor repo_id, subpath in ALL_REPOS:\n    print(f\"\\n--- {repo_id} ---\")\n    ok = download_hf(repo_id, subpath)\n    if ok:\n        ok_count += 1\n    else:\n        fail_count += 1\n        print(f\"  [WARN] {repo_id} basarisiz, devam ediliyor...\")\n\nprint(f\"\\n{'='*60}\")\nprint(f\"HF On-indirme: {ok_count}/{len(ALL_REPOS)} basarili\")\nif fail_count > 0:\n    print(f\"  {fail_count} repo indirilemedi — build'de tekrar denenebilir\")\nprint(f\"Disk : {disk_free()}GB free\")\nprint(f\"HF cache: {HF_CACHE_DIR}\")\nprint(f\"Drive: {DRIVE_DIR}\")

## Hucre 3: Docker Build (Model Bazli - Build + Drive Kayit + Temizlik)\n\nHer model:\n  1. Dockerfile FROM -> localhost:5000/ai-publisher-base:latest\n  2. Kaniko build (HF cache zaten Drive'da oldugu icin indirmeye gerek yok)\n  3. .tar.gz -> Drive\n  4. HF cache + Kaniko artifact temizle\n\nbuild_all_v2.sh'deki MODELS listesini kullaniyor.

In [ ]:
import subprocess, shutil, os, sys, time\nfrom pathlib import Path\n\nos.chdir(WORKSPACE)\n\nMODELS = [\n    \"cogvideox\", \"wan\", \"ltx\", \"hunyuan\", \"svd\",\n    \"animatediff\", \"wan25\", \"zeroscope\", \"dynamicrafter\",\n    \"xtts\", \"audioldm2\", \"f5tts\", \"kokorotts\", \"wav2lip\",\n    \"musetalk\", \"whisper\", \"stablediffusion\", \"sadtalker\",\n    \"pyramid-flow\", \"mochi\", \"video-retalking\", \"geneface\",\n    \"lora-trainer\",\n]\n\ndef ts():\n    return time.strftime(\"%H:%M:%S\")\n\ndef run(cmd, check=True, capture=False, timeout=None):\n    kwargs = dict(shell=True, text=True, capture_output=capture)\n    if timeout: kwargs[\"timeout\"] = timeout\n    r = subprocess.run(cmd, **kwargs)\n    if capture:\n        r._out = r.stdout.strip() if r.stdout else \"\"\n    if check and r.returncode != 0:\n        print(f\"[HATA] {cmd[:100]}\")\n        if r.stderr: print(r.stderr[:300])\n        return None\n    return r\n\ndef cleanup(model_name):\n    \"\"\"Build sonrasi disk temizligi.\"\"\"\n    print(f\"  [{ts()}] [CLEANUP] HF cache + Kaniko artifact\")\n    run(\"rm -rf /workspace/hf_cache/* /workspace/kaniko-* /tmp/kaniko-*\", check=False)\n    run(\"sync\", check=False)\n    print(f\"  [{ts()}] [CLEANUP] Disk: {disk_free()}GB free\")\n\ndef build_model(model):\n    model_dir = WORKSPACE / model\n    if not (model_dir / \"Dockerfile\").exists():\n        print(f\"  [{ts()}] [SKIP] {model}/Dockerfile yok\")\n        return \"skip\"\n    \n    drive_tar = DRIVE_DIR / f\"{model}.tar.gz\"\n    \n    # Drive'da gecerli build var mi? (sha256 kontrol)\n    # Basit: sadece yoksa veya ~< 100MB ise build et\n    if drive_tar.exists() and drive_tar.stat().st_size > 100 * 1024 * 1024:\n        size_mb = drive_tar.stat().st_size / 1024 / 1024\n        print(f\"  [{ts()}] [SKIP] {model} zaten var ({size_mb:.0f} MB)\")\n        return \"skip\"\n    \n    print(f\"  [{ts()}] [BUILD] {model}...\")\n    \n    # Drive'dan HF cache kullanmak icin HF_HOME set et\n    env = os.environ.copy()\n    env[\"HF_HOME\"] = str(HF_CACHE_DIR)\n    \n    # 1) FROM satiriyi yama ve ARG HF_TOKEN enjekte et\n    df_path = model_dir / \"Dockerfile\"\n    df_content = df_path.read_text()\n    patched_lines = []\n    for line in df_content.splitlines():\n        patched_lines.append(line)\n        if line.strip().startswith(\"FROM \"):\n            patched_lines.append(\"ARG HF_TOKEN\")\n    patched_content = \"\\n\".join(patched_lines)\n    if \"FROM ai-publisher-base:latest\" in patched_content:\n        patched_content = patched_content.replace(\"FROM ai-publisher-base:latest\",\n                                                  \"FROM localhost:5000/ai-publisher-base:latest\")\n    df_path.write_text(patched_content)\n    \n    # 2) shared/ varsa kopyala\n    shared_src = WORKSPACE / \"shared\"\n    if shared_src.exists():\n        run(f\"cp -rf {shared_src} {model_dir}/\", check=False)\n    \n    # 3) runpod_handler + download_weights kopyala\n    for f in [\"runpod_handler.py\", \"download_weights.sh\"]:\n        src = WORKSPACE / f\n        if src.exists():\n            run(f\"cp -f {src} {model_dir}/\", check=False)\n    \n    # 4) Kaniko build\n    hf_token = os.environ.get(\"HF_TOKEN\", \"\")\n    build_cmd = (\n        f\"{KANIKO} \"\n        f\"--context={model_dir}/ \"\n        f\"--dockerfile={model_dir}/Dockerfile \"\n        f\"--destination=localhost:5000/ai-publisher-{model}:latest \"\n        f\"--tarPath=/tmp/{model}.tar \"\n        f\"--build-arg HF_TOKEN='{hf_token}' \"\n        f\"--insecure --skip-tls-verify \"\n        f\"--whitelist-var-run=false --ignore-var-run \"\n        f\"--snapshot-mode=redo\"\n    )\n    \n    t0 = time.time()\n    p = subprocess.run(build_cmd, shell=True, text=True, env=env)\n    duration = time.time() - t0\n    \n    # 5) Dockerfile'i geri al\n    subprocess.run(f\"git checkout -- {df_path}\", shell=True)\n    \n    if p.returncode != 0:\n        print(f\"  [{ts()}] [FAIL] {model} build basarisiz ({duration:.0f}s)\")\n        cleanup(model)\n        return \"fail\"\n    \n    print(f\"  [{ts()}] [OK] Build tamam ({duration:.0f}s)\")\n    \n    # 6) .tar -> Drive'a .tar.gz olarak kaydet\n    tmp_tar = Path(f\"/tmp/{model}.tar\")\n    if tmp_tar.exists():\n        print(f\"  [{ts()}] [SAVE] Drive'a sıkıstiriyor...\")\n        t1 = time.time()\n        r = run(f\"pigz -c {tmp_tar} > {drive_tar}\", check=False, timeout=3600)\n        if r is None or r.returncode != 0:\n            # pigz yoksa gzip dene\n            r = run(f\"gzip -c {tmp_tar} > {drive_tar}\", timeout=3600)\n        \n        size_mb = drive_tar.stat().st_size / 1024 / 1024\n        print(f\"  [{ts()}] [OK] {drive_tar.name} ({size_mb:.0f} MB, {time.time()-t1:.0f}s)\")\n        tmp_tar.unlink(missing_ok=True)\n    \n    # 7) Temizlik\n    cleanup(model)\n    \n    return \"ok\"\n\nprint(f\"{len(MODELS)} model build edilecek.\")\nprint(f\"HF cache : {HF_CACHE_DIR}\")\nprint(f\"Drive    : {DRIVE_DIR}\")\nprint(f\"Disk     : {disk_free()}GB free\")\nprint()\n\nresults = {}\nfor i, model in enumerate(MODELS, 1):\n    print(f\"\\n{'='*60}\")\n    print(f\"[{i}/{len(MODELS)}] {model}\")\n    print(f\"    Disk: {disk_free()}GB free\")\n    results[model] = build_model(model)\n    time.sleep(1)\n\nok_m = [k for k,v in results.items() if v == \"ok\"]\nfail_m = [k for k,v in results.items() if v == \"fail\"]\nskip_m = [k for k,v in results.items() if v == \"skip\"]\n\nprint(f\"\\n{'='*60}\")\nprint(f\"SONUC: {len(ok_m)} basarili, {len(fail_m)} basarisiz, {len(skip_m)} atlandi\")\nif fail_m: print(f\"  Basarisiz: {fail_m}\")\nif skip_m: print(f\"  Atlandi : {skip_m}\")\nprint(f\"Disk     : {disk_free()}GB free\")

## Hucre 4: GHCR Push\n\nDrive'daki .tar.gz dosyalarini GHCR'a push eder.\nOnce Hucre 3 build tamamlanmali.

In [ ]:
# GHCR auth\nfrom google.colab import userdata\nimport base64, json, requests, sys\n\ngh_token = None\nfor key in [\"GITHUB_PAT\", \"GITHUB_TOKEN\", \"GH_TOKEN\"]:\n    try:\n        gh_token = userdata.get(key)\n        if gh_token:\n            print(f\"Token: {key}\")\n            break\n    except:\n        pass\n\nif not gh_token:\n    print(\"Token bulunamadi!\")\n    sys.exit(1)\n\ngh_token = gh_token.strip()\nghcr_user = \"arda-avci\"\n\n# Podman auth\n_auth_dir = Path(os.path.expanduser(\"~/.config/containers\"))\n_auth_dir.mkdir(parents=True, exist_ok=True)\n_auth_b64 = base64.b64encode(f\"{ghcr_user}:{gh_token}\".encode()).decode()\n(_auth_dir / \"auth.json\").write_text(\n    json.dumps({\"auths\": {\"ghcr.io\": {\"auth\": _auth_b64}}})\n)\nprint(\"GHCR auth: OK\")\n\nGHCR_NS = \"ghcr.io/arda-avci\"\nIMAGE_PREFIX = \"ai-publisher-\"\n\narchives = sorted(DRIVE_DIR.glob(\"*.tar.gz\"))\nprint(f\"{len(archives)} arsiv bulundu\")\n\nTEMP_DIR = Path(\"/content/tmp_push\")\nTEMP_DIR.mkdir(parents=True, exist_ok=True)\n\nsuccess = 0\nfailed = []\n\nfor arc in archives:\n    model = arc.stem.replace(\".tar\", \"\")\n    ghcr_img = f\"{GHCR_NS}/{IMAGE_PREFIX}{model}:latest\"\n    size_mb = arc.stat().st_size / 1024 / 1024\n    \n    print(f\"\\n[{ts()}] {model} ({size_mb:.0f} MB)\")\n    \n    tmp_tar = TEMP_DIR / f\"{model}.tar\"\n    \n    # Decompress\n    print(f\"  [{ts()}] Decompress...\")\n    r = run(f\"gunzip -c {arc} > {tmp_tar}\", check=False, timeout=7200)\n    if not tmp_tar.exists():\n        print(f\"  [HATA] Decompress basarisiz\")\n        failed.append(model); continue\n    \n    # Podman load\n    print(f\"  [{ts()}] Podman load...\")\n    r = run(f\"podman load -i {tmp_tar}\", check=False, timeout=7200)\n    if r is None or r.returncode != 0:\n        print(f\"  [HATA] Podman load\")\n        failed.append(model)\n        tmp_tar.unlink(missing_ok=True)\n        continue\n    \n    # Parse loaded name\n    loaded = None\n    for line in (r.stdout or \"\").split(\"\\n\"):\n        if \"Loaded image\" in line:\n            parts = line.split(\":\", 1)\n            loaded = parts[-1].strip()\n            if \"sha256:\" in loaded: loaded = None\n            break\n    if not loaded:\n        loaded = f\"localhost/{IMAGE_PREFIX}{model}:local\"\n    \n    # Push\n    print(f\"  [{ts()}] Podman push...\")\n    push_ok = False\n    for attempt in range(3):\n        r = run(f\"podman tag {loaded} {ghcr_img} && podman push {ghcr_img}\",\n                check=False, timeout=1800)\n        if r is not None and r.returncode == 0:\n            print(f\"  [{ts()}] [OK] Push tamam!\")\n            push_ok = True\n            break\n        print(f\"  [RETRY] deneme {attempt+1}/3\")\n        time.sleep(10)\n    \n    run(f\"podman rmi {loaded} 2>/dev/null\", check=False)\n    run(f\"podman rmi {ghcr_img} 2>/dev/null\", check=False)\n    tmp_tar.unlink(missing_ok=True)\n    \n    if push_ok:\n        success += 1\n    else:\n        failed.append(model)\n\nprint(f\"\\n{'='*60}\")\nprint(f\"SONUC: {success}/{len(archives)} GHCR'a push edildi\")\nif failed: print(f\"  Basarisiz: {failed}\")\nprint()\nprint(\"GHCR dogrulama: https://github.com/arda-avci?tab=packages\")